# 3DFist training (Kaggle or local)

Trains HandLocNet and the 3DFist pose model (`train_handloc.py`, `train_fist.py`), then evaluates
them (`eval_fist.py`, `lr_report.py`). The notebook first **measures this machine and predicts how
long the whole run takes**, before anything is trained.

The repo has no dataset or weights. Point the notebook at them:

**On Kaggle**
1. Upload `dataset_ir/` (the folder of `session_*` dirs, with `val_sessions.txt`) as a Kaggle dataset
   and attach it with *Add Input*. Optional: attach a dataset with `.pth` files (a `handloc*.pth` to
   skip HandLocNet training, or `<RUN_NAME>_last.pth` to resume).
2. Settings: *Accelerator* = GPU (P100 or T4), *Internet* = on (clones the code from GitHub and
   downloads the ImageNet ResNet-18 weights). If the code is attached as a dataset, it is used instead of cloning.
3. Leave `DATA_DIR = None`: the first attached folder that holds sessions is used.
   Run all. Weights and results end up in `/kaggle/working/3d-hand-reconstruction/{checkpoints,results}` (the Output tab).

**Locally:** open this notebook from the repo root and set `DATA_DIR` (and `CKPT_DIR` if you have weights).

Kaggle GPU sessions stop after 12 h. If the estimate is longer, lower `EPOCHS_FIST`, or run twice:
attach the first run's output, which has `<RUN_NAME>_last.pth`, and training resumes from it.

In [ ]:
# ---- Settings ----
DATA_DIR = None          # dataset_ir folder; None = search /kaggle/input (Kaggle) or ./dataset_ir, ../dataset_ir (local)
CKPT_DIR = None          # folder with .pth files to reuse; None = search /kaggle/input (Kaggle) or ./checkpoints (local)
CODE_DIR = None          # repo folder; None = this folder if it is the repo, else an attached copy, else git clone
REPO_URL = "https://github.com/Sangameshwar-1/3d-hand-reconstruction.git"
REPO_BRANCH = "main"

RUN_NAME = "fist_r18_v9"         # checkpoints/<RUN_NAME>_best.pth, results/<RUN_NAME>_training_log.csv
HANDLOC_NAME = "handloc_v9"      # checkpoints/<HANDLOC_NAME>_best.pth when HandLocNet is trained here
TRAIN_HANDLOC = None             # None = train only if no handloc*.pth is found; True / False to force
RESUME = True                    # continue from <RUN_NAME>_last.pth if one is found

# v8 recipe (README)
EPOCHS_HANDLOC = 30
EPOCHS_FIST = 60
BATCH = 8
LR = 1e-3
MAX_ROLL = 180
CURL_WEIGHT = 10
import os
WORKERS = 4 if os.name != "nt" else 0   # DataLoader workers; on Windows each worker would copy the whole dataset

# Calibration knobs for the time estimate
USABLE_FRACTION = 0.75   # share of frames that pass the label filters (README: 17,582 / 23,685)
WORKER_EFFICIENCY = 0.7  # Kaggle's 4 vCPUs are 2 hyper-threaded cores, so workers don't scale linearly

In [ ]:
import shutil, subprocess, sys, time
from pathlib import Path

KAGGLE = Path("/kaggle/input").is_dir()


def search(root, test, depth=4):
    """First folder under root (breadth by os.walk) for which test(path, subdirs, files) holds."""
    root = Path(root)
    if not root.is_dir():
        return None
    for d, subdirs, files in os.walk(root):
        p = Path(d)
        if test(p, subdirs, files):
            return p
        subdirs[:] = [] if len(p.relative_to(root).parts) >= depth else \
            [s for s in subdirs if s not in ("left", "right", "leap")]
    return None


# ---- Code ----
has_code = lambda p, subs, files: "train_fist.py" in files
if CODE_DIR is None:
    if (Path.cwd() / "train_fist.py").is_file():
        CODE_DIR = Path.cwd()
    else:
        CODE_DIR = Path("/kaggle/working/3d-hand-reconstruction" if KAGGLE else "3d-hand-reconstruction").resolve()
        attached = search("/kaggle/input", has_code) if KAGGLE else None
        if (CODE_DIR / "train_fist.py").is_file():
            pass
        elif attached:
            shutil.copytree(attached, CODE_DIR, dirs_exist_ok=True)  # /kaggle/input is read-only
        else:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CODE_DIR)],
                           check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
CODE_DIR = Path(CODE_DIR).resolve()
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))

# ---- Dataset ----
is_dataset = lambda p, subs, files: "left" in subs or any((p / s / "left").is_dir() for s in subs)
if DATA_DIR is None:
    roots = ["/kaggle/input"] if KAGGLE else [CODE_DIR / "dataset_ir", CODE_DIR.parent / "dataset_ir"]
    DATA_DIR = next((d for r in roots if (d := search(r, is_dataset))), None)
if DATA_DIR is None or not Path(DATA_DIR).is_dir():
    raise FileNotFoundError(f"No dataset found (DATA_DIR={DATA_DIR}). Attach dataset_ir on Kaggle or set DATA_DIR.")
DATA_DIR = Path(DATA_DIR).resolve()

# ---- Existing weights (ignore Git LFS pointer files, which are a few hundred bytes) ----
ckpt_root = CKPT_DIR or ("/kaggle/input" if KAGGLE else CODE_DIR / "checkpoints")
found = sorted(Path(ckpt_root).rglob("*.pth")) if Path(ckpt_root).is_dir() else []
found += sorted((CODE_DIR / "checkpoints").glob("*.pth"))  # written by an earlier run of this notebook
found = sorted({p for p in found if p.stat().st_size > 10_000}, key=lambda p: p.stat().st_mtime)
(CODE_DIR / "checkpoints").mkdir(exist_ok=True)

handloc_found = [p for p in found if p.name.startswith("handloc")]
if TRAIN_HANDLOC is None:
    TRAIN_HANDLOC = not handloc_found
if TRAIN_HANDLOC:
    HANDLOC_WEIGHTS = CODE_DIR / "checkpoints" / f"{HANDLOC_NAME}_best.pth"
elif handloc_found:
    HANDLOC_WEIGHTS = handloc_found[-1]
else:
    raise FileNotFoundError("TRAIN_HANDLOC=False but no handloc*.pth found")

import torch
RESUME_FROM, START_EPOCH = None, 1
last = [p for p in found if p.name == f"{RUN_NAME}_last.pth"]
if RESUME and last:
    RESUME_FROM = last[-1]
    START_EPOCH = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)["epoch"] + 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Code     : {CODE_DIR}")
print(f"Dataset  : {DATA_DIR}")
print(f"HandLoc  : {'train -> ' if TRAIN_HANDLOC else 'reuse '}{HANDLOC_WEIGHTS}")
print(f"3DFist   : {RUN_NAME}, epochs {START_EPOCH}..{EPOCHS_FIST}" + (f" (resuming {RESUME_FROM})" if RESUME_FROM else ""))
print(f"Device   : {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU only: training will be very slow'}"
      f" | CPUs {os.cpu_count()} | workers {WORKERS}")

## 1. Predicted run time

Measured on this machine, then scaled to the dataset:
- frame loading and crop rendering on one real session,
- one training step and one validation step of each model on the GPU, with synthetic batches of the real shapes,
- `eval_fist.py`, which runs on the CPU.

The frame counts come from the files on disk, multiplied by `USABLE_FRACTION`. The last cell prints the real time
next to this estimate, so you can tune the two knobs. For reference, v8 took about 2 h 20 min on a GTX 1080 Ti
(60 epochs including HandLocNet).

In [ ]:
import math, random
import numpy as np
import torch.nn.functional as F
from fist_dataset import find_sessions, StereoIRDataset
from fist_model import build_fist_model
from fist_loss import PhysicsInformedFistLoss
from hand_locator import HandLocNet, locator_loss, make_targets, prepare_view, decode
from ir_transforms import StereoPerspectivePreprocessor
from perspective_crop import GEO_DIM

dev = torch.device(DEVICE)
sync = torch.cuda.synchronize if DEVICE == "cuda" else (lambda: None)


def timeit(fn, n, warmup=3):
    for _ in range(warmup):
        fn()
    sync()
    t = time.perf_counter()
    for _ in range(n):
        fn()
    sync()
    return (time.perf_counter() - t) / n


# Frame counts, same session split as fist_dataset.split_sessions
sessions = find_sessions(DATA_DIR)
raw = {s.name: sum(1 for _ in (s / "left").glob("*.png")) for s in sessions}
total_raw = sum(raw.values())
if len(sessions) == 1:
    val_raw = 0.2 * total_raw
else:
    vf = DATA_DIR / "val_sessions.txt"
    val_names = set(vf.read_text().split()) if vf.exists() else {s.name for s in sessions[-max(1, round(len(sessions) * 0.2)):]}
    val_raw = sum(n for k, n in raw.items() if k in val_names)
tr_raw = total_raw - val_raw
n_tr, n_val = tr_raw * USABLE_FRACTION, val_raw * USABLE_FRACTION
print(f"{len(sessions)} sessions, {total_raw:,} frames on disk -> ~{n_tr:,.0f} train / ~{n_val:,.0f} val usable")

# Loading + crop rendering, on the smallest session with at least 100 frames that loads
for s in sorted(sessions, key=lambda s: raw[s.name]):
    if raw[s.name] < 100:
        continue
    try:
        t = time.perf_counter()
        ds = StereoIRDataset(s, split="train", max_roll_deg=MAX_ROLL, mirror_prob=0.5)  # one session as its own dataset
        t_load = (time.perf_counter() - t) / (0.8 * raw[s.name])
        break
    except (FileNotFoundError, RuntimeError):
        continue
else:
    raise RuntimeError("No session could be loaded: check that sessions have rect_lut.npz and calibration.json")
t_crop = timeit(lambda: ds[random.randrange(len(ds))], 50)
img_l, img_r, _, bbox_l, bbox_r, luts = ds.cached_data[0]
t_hl_data = timeit(lambda: (prepare_view(img_l), make_targets(bbox_l, img_l.shape[1])), 200)

# 3DFist steps (same model, loss and clipping as train_fist.py)
model = build_fist_model("resnet18", pretrained=False, hand_head=True).to(dev)
crit = PhysicsInformedFistLoss(lambda_xy=1.0, lambda_z=3.0, lambda_bone=1.5, lambda_angle=1.0,
                               lambda_curl=CURL_WEIGHT).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
x = torch.rand(BATCH, 2, 128, 128, device=dev) * 2 - 1
geo = torch.rand(BATCH, GEO_DIM, device=dev)
gt = torch.randn(BATCH, 21, 3, device=dev) * 50
gt = gt - gt[:, :1]
hand = torch.rand(BATCH, device=dev).round()


def fist_step():
    opt.zero_grad()
    p, h = model(x, geo, return_hand=True)
    loss, _ = crit(p, gt)
    (loss + F.binary_cross_entropy_with_logits(h, hand)).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    opt.step()


@torch.no_grad()
def fist_val():  # train_fist.evaluate_model also runs the mirrored batch
    p, _ = model(x, geo, return_hand=True)
    crit(p, gt)
    model(x.flip(-1).flip(-3), geo, return_hand=True)


model.train(); t_fist_step = timeit(fist_step, 20)
model.eval(); t_fist_val = timeit(fist_val, 20)

# HandLocNet steps (train_handloc.py: batch 32, DataLoader without workers)
HL_BATCH = 32
hl = HandLocNet().to(dev)
hopt = torch.optim.AdamW(hl.parameters(), lr=1e-3)
hx = torch.rand(HL_BATCH, 1, 128, 128, device=dev) * 2 - 1
heat, size = (torch.from_numpy(a)[None].repeat(HL_BATCH, 1, 1).to(dev) for a in make_targets(bbox_l, img_l.shape[1]))


def hl_step():
    loss = locator_loss(hl(hx), heat, size)
    hopt.zero_grad()
    loss.backward()
    hopt.step()


@torch.no_grad()
def hl_val():
    for o in hl(hx).cpu():
        decode(o, (384, 384))


@torch.no_grad()
def locate_pair():  # HandLocator.locate_pair on the training device (crop-jitter measurement)
    out = hl(torch.stack([prepare_view(img_l), prepare_view(img_r)]).to(dev)).cpu()
    decode(out[0], img_l.shape), decode(out[1], img_r.shape)


hl.train(); t_hl_step = timeit(hl_step, 20)
hl.eval(); t_hl_val = timeit(hl_val, 10); t_locate = timeit(locate_pair, 20)

# eval_fist.py runs on the CPU: locator + crop + model, one frame at a time
model_cpu, hl_cpu = model.to("cpu").eval(), hl.to("cpu").eval()
pre = StereoPerspectivePreprocessor(target_size=128, is_training=False)


@torch.no_grad()
def eval_frame():
    hl_cpu(torch.stack([prepare_view(img_l), prepare_view(img_r)]))
    xx, gg, _ = pre(img_l, img_r, luts, bbox_l, bbox_r)
    model_cpu(xx[None], gg[None])


t_eval = timeit(eval_frame, 10)
del ds, model, model_cpu, hl, hl_cpu, opt, hopt
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# Scale up
w = max(1.0, WORKERS * WORKER_EFFICIENCY)  # workers=0 renders crops in the main process
fist_epoch = (math.ceil(n_tr / BATCH) * max(t_fist_step, BATCH * t_crop / w)
              + math.ceil(n_val / BATCH) * max(t_fist_val, BATCH * t_crop / w))
hl_epoch = (math.ceil(2 * n_tr / HL_BATCH) * (t_hl_step + HL_BATCH * t_hl_data)
            + math.ceil(2 * n_val / HL_BATCH) * (t_hl_val + HL_BATCH * t_hl_data))
fist_epochs = max(0, EPOCHS_FIST - START_EPOCH + 1)
FIST_KEY = f"3DFist: {fist_epochs} epochs ({fist_epoch / 60:.1f} min each)"
plan = {
    "Clone / setup / weight downloads": 60,
    "HandLocNet: load frames": (tr_raw + val_raw) * t_load if TRAIN_HANDLOC else 0,
    f"HandLocNet: {EPOCHS_HANDLOC} epochs": EPOCHS_HANDLOC * hl_epoch if TRAIN_HANDLOC else 0,
    "3DFist: load frames + measure locator": (tr_raw + val_raw) * t_load + n_val * t_locate,
    FIST_KEY: fist_epochs * fist_epoch,
    "eval_fist.py (CPU)": (tr_raw + val_raw) * t_load + n_val * t_eval,
    "lr_report.py": val_raw * t_load + n_val * t_crop / w,
}
hms = lambda s: f"{int(s // 3600)}:{int(s % 3600 // 60):02d}:{int(s % 60):02d}"
print(f"\nMeasured: load {t_load * 1e3:.1f} ms/frame | crop {t_crop * 1e3:.1f} ms/sample | "
      f"3DFist step {t_fist_step * 1e3:.0f} ms, val {t_fist_val * 1e3:.0f} ms | "
      f"HandLocNet step {t_hl_step * 1e3:.0f} ms | eval {t_eval * 1e3:.0f} ms/frame (CPU)")
bound = "GPU" if t_fist_step >= BATCH * t_crop / w else "CPU crop rendering (more workers would help)"
print(f"3DFist training is limited by: {bound}\n")
for k, v in plan.items():
    print(f"  {k:<44s} {hms(v):>9s}")
PREDICTED_S = sum(plan.values())
print(f"  {'TOTAL (predicted)':<44s} {hms(PREDICTED_S):>9s}")
if KAGGLE and PREDICTED_S > 11.5 * 3600:
    spare = 11.5 * 3600 - (PREDICTED_S - plan[FIST_KEY])
    print(f"\nWARNING: over Kaggle's 12 h limit. About {max(0, int(spare / fist_epoch))} 3DFist epochs fit in one session; "
          f"lower EPOCHS_FIST, or let it stop and resume from {RUN_NAME}_last.pth in a second session.")
T_START = time.time()

In [ ]:
def run(script, *args, env=None):
    """Run a repo script and stream its output into the notebook."""
    cmd = [sys.executable, script, *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"{script} exited with code {p.returncode}")

## 2. HandLocNet (hand box in each raw IR view)

In [ ]:
if TRAIN_HANDLOC:
    run("train_handloc.py", "--dataset", DATA_DIR, "--epochs", EPOCHS_HANDLOC, "--device", DEVICE, "--out", HANDLOC_WEIGHTS)
else:
    print(f"Skipped: reusing {HANDLOC_WEIGHTS}")

## 3. 3DFist (stereo IR crops → 21 joints)

In [ ]:
if START_EPOCH > EPOCHS_FIST:
    print(f"Skipped: {RESUME_FROM} already finished {EPOCHS_FIST} epochs")
else:
    run("train_fist.py", "--dataset", DATA_DIR, "--epochs", EPOCHS_FIST, "--batch-size", BATCH, "--lr", LR,
        "--device", DEVICE, "--workers", WORKERS, "--handloc", HANDLOC_WEIGHTS, "--backbone", "resnet18",
        "--hand-head", "--max-roll", MAX_ROLL, "--balance-curl", "--mirror", "--curl-weight", CURL_WEIGHT,
        "--run-name", RUN_NAME, *(["--resume", RESUME_FROM] if RESUME_FROM else []))

## 4. Evaluation

In [ ]:
BEST = CODE_DIR / "checkpoints" / f"{RUN_NAME}_best.pth"
if not BEST.exists() and RESUME_FROM:  # resumed run that finished in an earlier session: its best checkpoint is next to the last one
    BEST = RESUME_FROM.with_name(f"{RUN_NAME}_best.pth")
EVAL_JSON = CODE_DIR / "results" / f"eval_{RUN_NAME}.json"
run("eval_fist.py", "--dataset", DATA_DIR, "--weights", BEST, "--handloc", HANDLOC_WEIGHTS, "--out", EVAL_JSON)
run("lr_report.py", BEST, env={"DATASET_IR": str(DATA_DIR)})

In [ ]:
import csv
import matplotlib.pyplot as plt
from IPython.display import Image, display

display(Image(str(EVAL_JSON.with_name(EVAL_JSON.name.replace("eval_", "eval_plot_").replace(".json", ".png")))))
log = list(csv.DictReader(open(CODE_DIR / "results" / f"{RUN_NAME}_training_log.csv")))
plt.figure(figsize=(7, 3.5))
plt.plot([int(r["epoch"]) for r in log], [float(r["val_mpjpe_mm"]) for r in log])
plt.xlabel("epoch"); plt.ylabel("val MPJPE (mm)"); plt.title(RUN_NAME); plt.grid(alpha=0.3)
plt.show()

actual = time.time() - T_START + plan["Clone / setup / weight downloads"]
print(f"Run time: {hms(actual)} actual vs {hms(PREDICTED_S)} predicted ({actual / PREDICTED_S:.2f}x)")
print(f"Outputs: {CODE_DIR / 'checkpoints'}  and  {CODE_DIR / 'results'}")